**Ô 1.** Mount Google Drive để giữ dữ liệu và bản build OpenPCDet qua các phiên.

In [ ]:
# Ô 1
from google.colab import drive
drive.mount("/content/drive")
import os
os.environ["VCF_HOME"] = "/content/drive/MyDrive/vcf"
os.makedirs(os.environ["VCF_HOME"], exist_ok=True)

**Ô 2.** Lấy mã nguồn (clone lần đầu, các lần sau `git pull`).

In [ ]:
# Ô 2
%cd /content
![ -d VCuboidFit ] && git -C VCuboidFit pull -q || git clone -q https://github.com/Nguyendev037/VCuboidFit.git
%cd /content/VCuboidFit

**Ô 3.** Cài môi trường bằng một lệnh. Phiên bị ngắt thì chạy lại ô này, script dùng cache trên Drive.

In [ ]:
# Ô 3
!bash model/scripts/colab_setup.sh
os.environ["PCDET_ROOT"] = os.environ["VCF_HOME"] + "/OpenPCDet"
os.environ["PYTHONPATH"] = "/content/VCuboidFit/model/worker:" + os.environ["PCDET_ROOT"]

**Ô 4.** Smoke test: chạy bộ test LiDAR không cần GPU để chắc môi trường ổn.

In [ ]:
# Ô 4
%cd /content/VCuboidFit/model/worker
!python -m pytest -q -m "not perf and not gpu" tests/lidar
%cd /content/VCuboidFit

**Ô 5.** (Tuỳ chọn) Tải nuScenes-mini vào Drive. Cần tài khoản nuScenes: tải tay từ nuscenes.org rồi giải nén vào `$VCF_HOME/nuscenes`; notebook KHÔNG chứa credential.

In [ ]:
# Ô 5
NUSC = os.environ["VCF_HOME"] + "/nuscenes"
EXP = os.environ["VCF_HOME"] + "/exp"
print("Thư mục dữ liệu:", NUSC, "| tồn tại:", os.path.isdir(NUSC))

**Ô 6.** Tầng 0 trên bản mini (CPU).

In [ ]:
# Ô 6
if not os.path.isdir(NUSC):
    print("Bỏ qua Ô 6: chưa có nuScenes trên Drive (Ô 5). Cách B: nhảy tới Ô 8.")
else:
    %cd /content/VCuboidFit/model/worker
    !python -m c4.cli.lidar_experiment --data-root $NUSC --out $EXP --split V --tune
    %cd /content/VCuboidFit

**Ô 7.** Tầng 1 trên bản mini (cần GPU, không dùng Docker).

In [ ]:
# Ô 7
if not os.path.isdir(NUSC):
    print("Bỏ qua Ô 7: chưa có nuScenes trên Drive (bước 5). Dùng Ô 8 chế độ agent.")
else:
    !bash model/scripts/tier1.sh "$NUSC" "$EXP" --no-docker --sweeps 1 --epochs 20 --batch 4

**Ô 8.** Chế độ agent (cách B): cần URL THẬT do `cloudflared` in ra trên máy chạy worker và token của worker. Chỉ cần đã chạy Ô 1–3. Hướng dẫn: `docs/colab-dev-setup.md` mục 4.

In [ ]:
# Ô 8
from getpass import getpass
import subprocess, sys, urllib.request
SERVER = input("URL tunnel của worker (https://<tên>.trycloudflare.com, lấy trong khung cloudflared): ").strip().rstrip("/")
if not SERVER.startswith("https://") or "xxxx" in SERVER:
    raise SystemExit("URL không hợp lệ: dán URL THẬT do cloudflared in ra trên máy chạy worker (xem docs/colab-dev-setup.md mục 4.4).")
try:
    print("Kiểm /health:", urllib.request.urlopen(SERVER + "/health", timeout=20).read().decode()[:120])
except Exception as e:
    raise SystemExit(f"Không gọi được {SERVER}/health ({e}). Kiểm cloudflared còn chạy và URL đúng (mục 4.4), đợi 30 s rồi chạy lại ô này.")
os.environ["VCF_REMOTE_TOKEN"] = getpass("VCF_REMOTE_TOKEN (dán vào, sẽ không hiện ký tự): ")
subprocess.run([sys.executable, "-u", "model/scripts/colab_agent.py", "--server", SERVER,
                "--work", "/content/vcf"], env=os.environ)